# Adel for developers, 0: overview of the package

**For:** developers who want to read, check, change or couple the model. **You need:** the user tutorial
`../user/01_first_run.ipynb` (what the model does), Python, numpy. **Run time:** a few seconds.

This series:

| | notebook | what it covers |
|---|---|---|
| 0 | `00_overview.ipynb` | the package's modules, the original Adel, how the port is validated, where the equations are |
| 1 | `01_shoot_architecture_in_metafspm.ipynb` | metafspm's component formalism on `ShootArchitecture`, Adel's tables, a canopy, the tillers, the coupling contract, the comparison with the original and the speed |
| 2 | `02_extend_the_component.ipynb` | add a variable and a process to the component, run and test it |

## 1. Where the model comes from

`openalea.adel` holds two things:

- **the original Adel**, frozen in `openalea.adel.legacy`: Adel-Wheat's dynamic architecture (`AdelDyn`, as cnwgrass
  drives it in Wheat-BRIDGES), its static version written in R (`Adel.R`, run through rpy2), plantgen, and their
  tools. It is the port's reference and is no longer changed;
- **the port**, `ShootArchitecture` (`openalea.adel.architecture`), Adel's shoot architecture as a metafspm
  component. It was first ported exactly, then reworked where the original had problems.

The docs' **port** pages describe it: `doc/port/equations.rst` (the model's rules), `doc/port/legacy.rst` (the
original's algorithms and every departure from them, each with its status: the AD-n items), `doc/port/performance.rst`
(the timings), `doc/port/components.rst` (every declaration).

The organs' growth (Adel-Wheat's kinetics) is not ported. In metafspm, development models give the organs' lengths;
`ScenarioDevelopment` (`openalea.adel.development`) is a simple linear scenario for that.

## 2. The modules

Each module's first docstring line (the port's; `legacy` and `plantgen` are the original's):

In [1]:
import importlib
import pkgutil

import openalea.adel as package

for module in pkgutil.iter_modules(package.__path__):
    if module.name in ("legacy", "plantgen"):
        continue
    doc = (importlib.import_module(f"openalea.adel.{module.name}").__doc__ or "").strip()
    print(f"{module.name:13} {doc.splitlines()[0] if doc else ''}")

architecture  Adel's shoot architecture as a metafspm StructuralComponent: the plants' topology in the MPG (axes, phytomers, organs,
comparison    The frozen legacy (openalea.adel.legacy) run on deterministic scenarios, and its records, to compare the port with.
data          
development   A simple deterministic development for ShootArchitecture: ScenarioDevelopment writes what a development and a
display       The shoot's colours for metafspm's matplotlib drawings (openalea.metafspm.display.draw): its elements green, or straw
kernels       Adel's numba kernels, in one place (metafspm's conventions.md, "Performance rules"): the turtle's interpretation of
leaf_shapes   Leaf shape databases for ShootArchitecture: each shape class' width profile r(s) along the blade (s from 0 at the base
simulation    Adel's wheat shoots for users: a plant grown in a few lines, its results as tables.
tables        The tables ShootArchitecture reads: Adel's axes table (axeT: one row per axis of each plant

- `architecture.py` holds the component, `ShootArchitecture`: its declarations and its steps. Tutorial 1 of this series
  reads it.
- `development.py` gives the organs' lengths (`ScenarioDevelopment`), the plant model `ScenarioShoot` coupling the two,
  and `shoot_scene` to build a scene.
- `tables.py` reads Adel's tables, or generates them from a few numbers. `leaf_shapes.py`, `blade_mesh.py` and
  `turtle.py` make the leaves' shapes, their meshes and the plant's placement.
- `comparison/` runs the original and the port on the same scenario, for the tests.
- `simulation.py` is the users' entry point: `simulate`, the summary and the tables.

## 3. How the port is validated

The tests run with `pytest test` from the repository's root (R must be on the PATH for the original's). They are in
three groups:

- `test/legacy/`: the original's own tests, run on the frozen original;
- `test/comparison/`: the port against the original, on the same scenario (`openalea.adel.comparison`): the structure,
  the elements' lengths and areas, the leaf shapes, the placement and every triangle, step by step, on one plant and on
  a stand. They agree to 10⁻¹⁰ m;
- `test/`: the port's own behaviour, where the original has no counterpart.

The comparison and port tests, each file's subject:

In [2]:
import ast
from pathlib import Path

for path in sorted(Path("../../../test").glob("test_*.py")) + sorted(Path("../../../test/comparison").glob("test_*.py")):
    doc = ast.get_docstring(ast.parse(path.read_text())) or ""
    print(f"{path.parent.name + '/' + path.name:42} {' '.join(doc.split())[:110]}")

test/test_anchored.py                      ShootArchitecture on a plant another model shares: a crown initiator (a stand-in for the use case's: a seed an
test/test_plantgen.py                      
test/test_simulation.py                    The users' entry point (openalea.adel.simulation): its summary agrees with the plant model run by shoot_scene,
test/test_standalone.py                    ShootArchitecture without given tables: Adel's sample plant by default, plants generated from a few scalars (t
comparison/test_elements.py                ShootArchitecture's elements against the legacy's (AdelDyn.update_geometry's update_organ_elements), on the ha
comparison/test_geometry.py                ShootArchitecture's placement and triangles against the legacy's turtle (mtg_interpreter.AdelVisitor) and mesh
comparison/test_leaf_shapes.py             LeafShapes' fitted midribs and age classes against the legacy's (geometric_elements.Leaves: fitting.fit_leaves
comparison/test_legacy_harness.p

Where the port departs from the original, the test states the rule it restores, and `doc/port/legacy.rst` registers it.
Corrections that change the original's results are options, off by default, so that the comparison holds: for
example `tiller_inclination` (inclined tillers, from static Adel) or `own_axis_top` (AD-37).

To rerun one file while working: `pytest test/comparison/test_structure.py -q`.

## 4. What to read next

- `01_shoot_architecture_in_metafspm.ipynb` reads the component's code through metafspm's component formalism, then
  builds a canopy, its tillers, the coupling contract, the comparison with the original and the speed.
- `02_extend_the_component.ipynb` adds a variable and a process.
- metafspm's own tutorial (`examples/soil_plant_atm_fspm2026_workshop/tutorial_fspm2026_general_usage.ipynb` in the metafspm
  repository) builds components and couples them step by step. Its user guide, `docs/user.md`, describes every
  part of the framework.